# Dataset Preparation Notebook Summary - ZIPFILE VER

This notebook prepares the **WIN32 subset** of the **EMBER2024** dataset from Hugging Face for downstream machine learning experiments. It creates four Parquet files that separate the data into malware detection and malware behavior prediction tasks.

> **Note:** This notebook downloads approximately **8.5 GB** of WIN32 data from the EMBER2024 dataset on Hugging Face. Make sure you have at least **20-30 GB of available disk space** before running the notebook to accommodate the downloaded data and generated Parquet files.

## Data Source

Dataset:
- https://huggingface.co/datasets/joyce8/EMBER2024

Only the **WIN32** samples are used.

## Dataset Sampling

To reduce storage and preprocessing time while maintaining reproducibility:

- A **deterministic random 20% sample** of the WIN32 **training** data is selected.
- The **entire WIN32 test set** is retained.
- Using a fixed random seed ensures the same subset can be regenerated.

This produces:

- Training set (20% deterministic sample)
- Test set (100%)

## Generated Datasets

The notebook creates four Parquet files:

### Malware Detection

Used for binary malware classification.

Training:
- `detection_train.parquet`

Testing:
- `detection_test.parquet`

Each row contains:

| Column | Description |
|---------|-------------|
| `sha256` | Sample SHA-256 hash |
| `label` | Malware label (0 = benign, 1 = malware) |
| `general` | JSON-serialized PE metadata |
| `strings` | JSON-serialized extracted strings/features |
| `imports` | JSON-serialized imported functions/libraries |

Nested dictionaries are serialized into JSON strings before being written to Parquet to ensure a consistent schema.

---

### Malware Behavior Identification

Used for malware family and behavior prediction.

Training:
- `behavior_train.parquet`

Testing:
- `behavior_test.parquet`

Each row contains:

| Column | Description |
|---------|-------------|
| `sha256` | Sample SHA-256 hash |
| `label` | Constant value `1` (only malware samples are included) |
| `family` | Malware family name |
| `behavior` | Reported malware behaviors |
| `mbc` | JSON-serialized Malware Behavior Catalog (MBC) techniques |
| `ttps` | JSON-serialized ATT&CK TTP identifiers |

The behavior dataset only contains malware samples, so the label is fixed to `1`.

## Output

Running the notebook produces four Parquet files:

```
win32_detection_train_20pct.parquet
win32_behavior_train_20pct.parquet
win32_test_detection.parquet
win32_test_behavior.parquet
```

These files provide reproducible, cleaned datasets that can be loaded directly into downstream machine learning pipelines without requiring additional preprocessing of the original Hugging Face dataset.

In [2]:
# install
!pip install -q gdown

In [12]:
# imports
from pathlib import Path
import gdown
import zipfile
import pandas as pd
import shutil

In [14]:
# find repo root (parent of notebooks/)
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
DATA_DIR.mkdir(exist_ok=True)

file_id = "1UwYNfKhweCbJHHMCwY9FO5ha77FcZUGo"
zip_path = ROOT / "win32_data.zip"

# download iff zipfile DNE
if not zip_path.exists():
    gdown.download(
        id=file_id,
        output=str(zip_path),
        quiet=False,
    )
# Check what was downloaded
print("Downloaded file size:", zip_path.stat().st_size, "bytes")


Downloading...
From (original): https://drive.google.com/uc?id=1UwYNfKhweCbJHHMCwY9FO5ha77FcZUGo
From (redirected): https://drive.google.com/uc?id=1UwYNfKhweCbJHHMCwY9FO5ha77FcZUGo&confirm=t&uuid=ec3866fb-288b-4633-badf-a9217af141ae
To: /Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/win32_data.zip
100%|██████████| 2.21G/2.21G [03:22<00:00, 10.9MB/s]

Downloaded file size: 2206627030 bytes


In [15]:
# extract into data/
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

print(f"Dataset extracted to {DATA_DIR}")

Dataset extracted to /Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/win32_data


In [16]:
# Cleanup
# if extraction created windata/data/, move parquet files up
nested_data = DATA_DIR / "data"

if nested_data.exists():
    for parquet_file in nested_data.glob("*.parquet"):
        shutil.move(str(parquet_file), str(DATA_DIR / parquet_file.name))

    # remove empty nested data folder
    shutil.rmtree(nested_data)

# Remove macOS metadata folder
macos_folder = DATA_DIR / "__MACOSX"
if macos_folder.exists():
    shutil.rmtree(macos_folder)

# Remove ZIP after successful extraction
zip_path.unlink()

print("Dataset ready:")
for f in DATA_DIR.glob("*.parquet"):
    print(f.name)

Dataset ready:
win32_test_detection.parquet
win32_detection_train_20pct.parquet
win32_test_behavior.parquet
win32_behavior_train_20pct.parquet
